## setup

In [1]:
import os

os.environ['APIHOST'] = 'api.openai.com'
with open('/root/openai_api_key','r') as f:
    os.environ['APIKEY'] = f.read().replace('\n','')
os.environ['APIMODEL'] = 'gpt-4o-mini'

## modified EoH

In [2]:
# UNDER DEVELOPMENT
from reallm.llm_solution_engine_v2 import EngineV2


from reallm import Solution, ProblemSpec
import os
import re


engine = EngineV2(
    api_spec=dict(
        host=os.environ['APIHOST'],
        key=os.environ['APIKEY'],
        model=os.environ['APIMODEL'],
        timeout=20
    ),
    problem_spec=None
)


class ThoughtfulSolution(Solution):
    def __init__(self, response: str, *args, **kwargs):
        super().__init__(response, *args, **kwargs)
        self.thought = (re.findall(r'<<(.*)>>', response) or [None])[-1]


    def __str__(self):
        parstr = super().super().__str__()
        header_str, eval_str = parstr.split('\n')
        return f"""
{header_str}

{self.thought}

```python
{self.program}
```
{eval_str}
"""

 

In [3]:
       

## PROMPT->SOLUTION OPERATIONS

def i1_solution():
    prmpt = f'''{engine._problem_spec.description}
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed <<>>. 
2. Next, implement the following Python function:
{engine._problem_spec.template}
Enclose within a code block like ```python ... ```. Do not give additional explanations.'''
    return engine.generate_artifact(prmpt, ThoughtfulSolution)


def e1_solution(*parent_solns):
    prmpt = f'''{engine._problem_spec.description}
I have {len(parent_solns)} existing algorithms with their codes as follows:
{'\n'.join([f'Solution num {i}:\n{ss.program}\n' for i, ss in enumerate(parent_solns)])}
Please help me create a new algorithm that has a totally different form from the given ones. 
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed <<>>.
2. Next, implement the following Python function:
{engine._problem_spec.template}
Enclose within a code block like ```python ... ```. Do not give additional explanations.'''
    return engine.generate_artifact(prmpt, ThoughtfulSolution)


def e2_solution(*parent_solns):
    prmpt = f'''{engine._problem_spec.description}
I have {len(parent_solns)} existing algorithms with their codes as follows:
{'\n'.join([f'Solution num {i}:\n{ss.program}\n' for i, ss in enumerate(parent_solns)])}
Please help me create a new algorithm that has a totally different form from the given ones but can be motivated from them.
1. Firstly, identify the common backbone idea in the provided algorithms. 
2. Secondly, based on the backbone idea describe your new algorithm in one sentence. The description must be inside within boxed <<>>.
3. Thirdly, implement the following Python function:
{engine._problem_spec.template}
Enclosed within a code block like ```python ... ```. Do not give additional explanations.'''
    return engine.generate_artifact(prmpt, ThoughtfulSolution)


def m1_solution(parent_soln):
    prmpt = f'''{engine._problem_spec.description}
I have one algorithm with its code as follows. Algorithm description:
{parent_soln.thought}
Code:
{str(parent_soln.program)}
Please assist me in creating a new algorithm that has a different form but can be a modified version of the algorithm provided.
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed <<>>.
2. Next, implement the following Python function:
{engine._problem_spec.template}
Enclosed within a code block like ```python ... ```. Do not give additional explanations.'''
    return engine.generate_artifact(prmpt, ThoughtfulSolution)


def m2_solution(parent_soln):
    prmpt = f'''{engine._problem_spec.description}
I have one algorithm with its code as follows. Algorithm description:
{parent_soln.thought}
Code:
{str(parent_soln.program)}
Please identify the main algorithm parameters and assist me in creating a new algorithm that has a different parameter settings of the score function provided.
1. First, describe your new algorithm and main steps in one sentence. The description must be inside within boxed <<>>.
2. Next, implement the following Python function:
{engine._problem_spec.template}
Enclosed within a code block like ```python ... ```. Do not give additional explanations.'''
    return engine.generate_artifact(prmpt, ThoughtfulSolution)

In [4]:
## EOH MAIN LOOP

import numpy as np
import math
from typing import List
from threading import Lock, Thread



class Population:
    def __init__(self, init_members=[], popsize=10, max_samples=100, max_generations=100):
        self.members: List[Solution] = init_members
        self.popsize = popsize
        self.lock = Lock()
        self.max_samples, self.max_generations = max_samples, max_generations
        self.n_samples, self.n_generations = len(self.members), 0
    
    def selection(self, k):
        with self.lock:
            funcs = [f for f in self.members if not math.isinf(f.evaluation.fitness)]
            if funcs == []:
                return [np.random.choice(self.members) for _ in range(k)]
            else:
                func = sorted(funcs, key=lambda f: f.evaluation.fitness, reverse=False)
                p = np.array([1 / (r + len(func)) for r in range(len(func))])
                p /= np.sum(p)
                return [np.random.choice(func, p=p) for _ in range(k)]

    def survival_filter(self):
            self.members = sorted(self.members, key=lambda f: f.evaluation.fitness, reverse=False)[:self.popsize]
    
    def register(self, solns):
        with self.lock:
            for soln in solns:
                self.members.append(soln)
                self.n_samples += 1
                if len(self.members) > self.popsize:
                    self.survival_filter()
                    self.n_generations += 1
    
    def eoh_completed(self):
        with self.lock:
            return self.n_samples > self.max_samples or self.n_generations > self.max_generations


def _eoh_worker(population: Population, e1_per_gen=1 ,e2_per_gen=1, m1_per_gen=1, m2_per_gen=1, num_e1_parents=2, num_e2_parents=2):
    while not population.eoh_completed():
        for _ in range(e1_per_gen):
            population.register([e1_solution(*population.selection(num_e1_parents))])
        for _ in range(e2_per_gen):
            population.register([e2_solution(*population.selection(num_e2_parents))])
        for _ in range(m1_per_gen):
            population.register([m1_solution(*population.selection(1))])
        for _ in range(m2_per_gen):
            population.register([m2_solution(*population.selection(1))])


def eoh(num_initial_samples=3, pop_size=5, max_samples=10, max_generations=10, problem_spec: ProblemSpec=None, n_workers=1, **kwargs) -> Solution:

    # this syntactical sugar just lets you specify an optional new problemspec to eoh(..)
    # specifying the problem spec in the engine constructor is also sufficient
    with engine.set_problem(problem_spec):  

        population = Population(
            [i1_solution() for _ in range(num_initial_samples)],
            max_generations=max_generations,
            max_samples=max_samples,
            popsize=pop_size,
        )
        sampler_threads = [
            Thread(target=_eoh_worker, args=(population,), kwargs=kwargs)
            for _ in range(n_workers)
        ]
        for t in sampler_threads:
            t.start()
        for t in sampler_threads:
            t.join()

        return min(population.members, key=lambda f: f.fitness)

## run EoH, get solution traces to experiment with

In [5]:
from reallm_problems.reevo import TSPACOProblemSpec, BPPoffACOProblemSpec, CVRPACOProblemSpec, MKPACOProblemSpec, OPACOProblemSpec

ALL_TASKS = dict(
    # OBP=(OBPEvaluation()),
    # TSP=(TSPEvaluation()),
    # JSSP=(JSSPEvaluation()),
     TSPACO=(TSPACOProblemSpec()), # WORKING
     BPPoffACO=(BPPoffACOProblemSpec()), # WORKING
     CVRPACO=(CVRPACOProblemSpec()), # WORKING
     MKPACO=(MKPACOProblemSpec()), # WORKING
     OPACO=(OPACOProblemSpec()), # WORKING
)


In [6]:
from reallm import ProblemSpec, Evaluation
import re



engine.start_trace()

res = eoh(
    problem_spec=ALL_TASKS['TSPACO'],
    n_workers=5
)

tg, tl = engine.dump_trace()
clean_res = {
    'bestsoln': res,
    'tracegraph': tg,
    'tracelist': tl
}
    

## IdeaCuration experiments

In [48]:

from typing import Optional
from reallm.base.artifact import Artifact


class IdeaInspirations(Artifact):
    def __init__(self, response, *args, **kwargs):
        super().__init__(response, *args, **kwargs)
        self.ideas = re.findall(r'<<(.*)>>', response)

def scrape_ideas(parent_soln: ThoughtfulSolution) -> IdeaInspirations:
    prmpt = f'''{engine._problem_spec.description}
I have one algorithm with its code as follows. Algorithm description:
{parent_soln.thought}
Code:
{str(parent_soln.program)}

Please describe a list of features included in this algorithm.
A 'feature' should be a simple to describe mechanical concept that could be implemented in any solution to this problem.
Features should not pertain to code level optimizations, packages or syntax suggestions. Features should be mechanically specific.
Do not use variable names specific to the example.
Provide your list of ideas with each idea in one line, bracketted like <<idea description>>.
'''
    return engine.generate_artifact(prmpt, IdeaInspirations, parents=[parent_soln])


class IdeaArchive(Artifact):
    # making this an artifact s.t. we get engine level tracing
    def __init__(self, response, *args, **kwargs):
        super().__init__(response, *args, **kwargs)
        new_ideas = re.findall(r'<<(.*)>>', response)
        self.old_ideas = []
        self.new_ideas = new_ideas

    @property
    def ideas(self):
        return self.old_ideas + self.new_ideas

class ProposedIdea(Artifact):
    def __init__(self, response, *args, **kwargs):
        super().__init__(response, *args, **kwargs)
        self.idea = (re.findall(r'<<(.*)>>', response) or [None])[0]


def propose_idea(new_ideas: IdeaInspirations, old_archive: Optional[IdeaArchive]) -> ProposedIdea:
    # if old_archive is None:
    #     archive = IdeaArchive('\n'.join([f'<<{i}>>' for i in new_ideas.ideas]))
    #     engine.register_artifact(archive, parents=[new_ideas])
    #     return archive

    if old_archive is None:
        old_ideas = []
    else:
        old_ideas = old_archive.ideas
    
    prmpt = f"""{engine._problem_spec.description}
```python
# FUNCTION SIGNATURE
{engine._problem_spec.template}
```

A 'feature' should be a simple to describe mechanical concept that could be implemented in any solution to this problem.
Features do not pertain to code level optimizations, packages or syntax suggestions. Features are mechanically specific.
The following features have been previously collected to help design a solution.
```
{'\n'.join([f'- {i}' for i in old_ideas])}
```

New features have been proposed, but we want to make sure we only include original ideas which are meaningfully distinct from what we collected previously.
```
{'\n'.join([f'- {i}' for i in new_ideas.ideas])}
```

From the newly provided features, please indicate *at most one* which is meaningfully distinct from what we already had.
If none of them are meaningfully distinct, do not provide any text in <<>>. When the original list is long, there will *usually* not be new ideas to indicate.
You can indicate features by retyping them *exactly as provided above*, on its own line, within brackets like <<text from useful idea>>.

"""
    proposed_idea = engine.generate_artifact(prmpt, ProposedIdea)#, parents=[new_ideas])
    return proposed_idea

class ProposedIdeaValidation(Artifact):
    def __init__(self, response, *args, **kwargs):
        super().__init__(response, *args, **kwargs)
        self.verdict = (re.findall(r'<<(.*)>>', response) or [None])[0]

def validate_proposed_idea(proposed_idea: ProposedIdea, old_archive: Optional[IdeaArchive]) -> ProposedIdeaValidation:
    
    if old_archive is None:
        old_ideas = []
    else:
        old_ideas = old_archive.ideas
    prmpt = f"""{engine._problem_spec.description}
```python
# FUNCTION SIGNATURE
{engine._problem_spec.template}
```

A 'feature' should be a simple to describe mechanical concept that could be implemented in any solution to this problem.
Features do not pertain to code level optimizations, packages or syntax suggestions. Features are mechanically specific.
The following features have been previously collected to help design a solution.
```
{'\n'.join([f'- {i}' for i in old_ideas])}
```

Now a new potential feature has been proposed:
{proposed_idea.idea}

We want to make sure we only include feature ideas which are (1) high quality, (2) distinct from the others, (3) not syntax or library dependent.
Under those criteria, is the newly proposed idea a good one? Explain your reasoning, and then provide a final verdict by returning <<Y>> or <<N>>
"""
    return engine.generate_artifact(prmpt, ProposedIdeaValidation, parents=[proposed_idea])#, old_archive]

class IdeaCurator:
    def __init__(self, initial=[]):
        self.ideas = initial
        self.latest_archive = None

    def catalogue_solution_ideas(self, soln):
        scraped_ideas: IdeaInspirations = scrape_ideas(soln)
        proposed_idea: ProposedIdea = propose_idea(scraped_ideas, self.latest_archive)
        if proposed_idea.idea is not None:
            idea_validation: ProposedIdeaValidation = validate_proposed_idea(proposed_idea, self.latest_archive)
            if idea_validation.verdict.upper() == 'Y':
                if self.latest_archive is not None:
                    old_ideas = (self.latest_archive.ideas)
                else:
                    old_ideas = []
                archive = IdeaArchive('\n'.join([f'<<{i}>>' for i in old_ideas+[proposed_idea.idea]]))
                engine.register_artifact(archive)#, parents=[proposed_idea, self.latest_archive])
                self.latest_archive = archive
            return (scraped_ideas, proposed_idea, idea_validation, self.latest_archive)
        return (scraped_ideas, proposed_idea, None, self.latest_archive)

    # def _update_archive(self, new_ideas):
    #     self.latest_archive = update_idea_archive(new_ideas, self.latest_archive)
    #     return self.latest_archive

# curator = IdeaCurator()

# ex_soln = clean_res['tracelist'][-2]
# with engine.set_problem(ALL_TASKS['TSPACO']):
#     s_i, l_a = curator.catalogue_solution_ideas(ex_soln)


In [49]:
%%time

curator = IdeaCurator()

all_arts = []
with engine.set_problem(ALL_TASKS['TSPACO']):
    for soln_idx, soln in enumerate(clean_res['tracelist']):
        all_arts.append(curator.catalogue_solution_ideas(soln))


CPU times: user 1.07 s, sys: 75.6 ms, total: 1.14 s
Wall time: 4min 3s


In [76]:
all_arts[0][-1].ideas

['Reflect edge attractiveness quantitatively by indicating relative desirability of each connection in the solution space.']

In [77]:
all_arts[5][-1].ideas

['Reflect edge attractiveness quantitatively by indicating relative desirability of each connection in the solution space.',
 'Calculate the inverse values of distances to prioritize shorter paths over longer ones.',
 'Incorporate an indicator for edges that are most critical for solution feasibility.',
 'Calculate the average distance of each node to all other nodes to assess overall connectivity.',
 'Incorporate handling of missing or undefined distances to maintain robustness in calculations.']

In [78]:
all_arts[15][-1].ideas

['Reflect edge attractiveness quantitatively by indicating relative desirability of each connection in the solution space.',
 'Calculate the inverse values of distances to prioritize shorter paths over longer ones.',
 'Incorporate an indicator for edges that are most critical for solution feasibility.',
 'Calculate the average distance of each node to all other nodes to assess overall connectivity.',
 'Incorporate handling of missing or undefined distances to maintain robustness in calculations.',
 'Calculate the maximum values of each row in the distance matrix to identify the most costly connections for each node.',
 'Compute pairwise interactions between edges to assess overall network structure.',
 'Calculate scores using the negative exponential to favor shorter distances.',
 'Compute the minimum distance for each node to highlight the most accessible connections.',
 'Set a threshold to handle cases where minimum distances are zero, ensuring score stability.',
 'Normalize scores a

In [85]:
all_arts[18][1].idea

'Output a heuristic matrix of the same shape as the input distance matrix for consistency.'

In [84]:
print(all_arts[18][2].raw_resp)

The proposed feature, "Output a heuristic matrix of the same shape as the input distance matrix for consistency," is indeed a relevant consideration in function design. Maintaining consistent shapes between the input and output can simplify integration with other parts of a system, ensure alignment with expectations, and make the output easier to work with when implementing subsequent processing steps.

However, this feature is more of a design requirement or constraint rather than a mechanical concept or heuristic. It doesn't add a new form of heuristic computation or provide any novel insights or methods of evaluating the edges based on their distance. Instead, it primarily serves to ensure that the output conforms to a specific structure rather than enhancing the heuristic information.

Given that this feature does not introduce a distinct heuristic mechanism and overlaps with general programming practices regarding function design, it does not meet the criteria for being a high-qua

In [50]:
for a_idx, a in enumerate(all_arts):
    print(a_idx, len(a[-1].ideas))

0 1
1 2
2 3
3 4
4 5
5 5
6 6
7 7
8 8
9 9
10 9
11 10
12 11
13 12
14 13
15 14
16 15
17 16
18 16
19 17
20 18
21 18
22 19


In [47]:
all_arts[-1][-1].ideas

['Consider both incoming and outgoing distances to gauge the attractiveness of edges from multiple perspectives.',
 'Calculate the reciprocal of distances to assess edge importance',
 'Calculate the average distance from each node to all other nodes to assess overall connectivity.',
 'Implement a scoring mechanism that reflects both individual node importance and inter-node relationships.',
 'Normalization of values to a common scale to facilitate comparison across different inputs.',
 'Calculate maximum values from rows to assess edge importance based on row constraints.',
 'Facilitate comparative analysis of edge contributions by considering both local (to specific edges) and global (to all edges) distance metrics.',
 '- Incorporate distance inversions to highlight shorter paths and maximize edge selection.',
 '- Compute minimum distances for each node to its nearest neighbor to assess closeness.',
 'Transformation of distance metrics into probabilistic scores for decision-making',
 

In [ ]:
## it really looks like GPT's dont like removing things from lists

In [20]:
s_i.ideas

['Compute the inverse of distances to prioritize shorter edges over longer ones.',
 'Implement an exponential decay function to reduce the impact of longer distances more drastically.',
 "Use numpy's error handling to manage divisions by zero or invalid operations gracefully.",
 'Normalize heuristic scores to a defined range (0 to 1) for consistent interpretation of edge promisingness.',
 'Utilize masking to ignore zero distances in computations to avoid influencing the heuristic scores.',
 'Employ a method to extract the maximum value from the computed heuristic scores for normalization purposes.',
 'Ensure that the output remains in the same shape as the input distance matrix for compatibility with downstream processes.',
 'Use numpy for efficient array operations, taking advantage of vectorization to handle large distance matrices.']

In [21]:
l_a.raw_resp

"<<Compute the inverse of distances to prioritize shorter edges over longer ones.>>\n<<Implement an exponential decay function to reduce the impact of longer distances more drastically.>>\n<<Use numpy's error handling to manage divisions by zero or invalid operations gracefully.>>\n<<Normalize heuristic scores to a defined range (0 to 1) for consistent interpretation of edge promisingness.>>\n<<Utilize masking to ignore zero distances in computations to avoid influencing the heuristic scores.>>\n<<Employ a method to extract the maximum value from the computed heuristic scores for normalization purposes.>>\n<<Ensure that the output remains in the same shape as the input distance matrix for compatibility with downstream processes.>>\n<<Use numpy for efficient array operations, taking advantage of vectorization to handle large distance matrices.>>"

In [22]:
l_a.ideas

['Compute the inverse of distances to prioritize shorter edges over longer ones.',
 'Implement an exponential decay function to reduce the impact of longer distances more drastically.',
 "Use numpy's error handling to manage divisions by zero or invalid operations gracefully.",
 'Normalize heuristic scores to a defined range (0 to 1) for consistent interpretation of edge promisingness.',
 'Utilize masking to ignore zero distances in computations to avoid influencing the heuristic scores.',
 'Employ a method to extract the maximum value from the computed heuristic scores for normalization purposes.',
 'Ensure that the output remains in the same shape as the input distance matrix for compatibility with downstream processes.',
 'Use numpy for efficient array operations, taking advantage of vectorization to handle large distance matrices.']